https://langchain-ai.github.io/langgraph/how-tos/react-agent-from-scratch/#define-graph-state

In [1]:
from langchain_ollama import ChatOllama

In [2]:
llm = ChatOllama(
    # model="granite4:350m",
    model='llama3.2:3b',
    validate_model_on_init=True,
    temperature=0,
)

In [3]:
from typing import (
    Annotated,
    Sequence,
    TypedDict,
)

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """The state of the agent."""

    # add_messages is a reducer
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [4]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int):
    '''
    Adds a and b.
    '''
    print(f'--> add a: {a} b: {b}')
    return a+b

@tool
def substract(a: int, b: int):
    '''
    Subtract a from  b.
    '''
    print(f'--> sub a: {a} b: {b}')
    return a-b

@tool
def multiply(a: int, b: int):
    '''
    Multiplies a and b.
    '''
    print(f'--> mult a: {a} b: {b}')
    a*b

@tool
def devides(a: int, b: int):
    '''
    Devides a and b.
    '''
    print(f'--> dev a: {a} b: {b}')
    a / b

tools = [add, substract, multiply, devides]

model_with_tools = llm.bind_tools(tools)

In [5]:
import json
from langchain_core.messages import ToolMessage, SystemMessage
from langchain_core.runnables import RunnableConfig

tools_by_name = {tool.name: tool for tool in tools}

# Define our tool node
def tool_node(state: AgentState):
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=json.dumps(tool_result),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs}


# Define the node that calls the model
def call_model(
    state: AgentState,
    config: RunnableConfig,
):

    system_prompt = SystemMessage(
        "You are a helpful assistant tasked with performing arithmetic on a set of inputs"
    )
    response = model_with_tools.invoke([system_prompt] + state["messages"], config)
    # We return a list, because this will get added to the existing list
    return {"messages": [response]}

def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    # If there is no function call, then we finish
    if not last_message.tool_calls:
        return "end"
    # Otherwise if there is, we continue
    else:
        return "continue"

In [6]:
from langgraph.graph import StateGraph, END
from IPython.display import Image, display

graph = StateGraph(AgentState)

graph.add_node("agent", call_model)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
   
    {
        # If `tools`, then we call the tool node.
        "continue": "tools",
        # Otherwise we finish.
        "end": END,
    },
)

graph.add_edge("tools", "agent")
app = graph.compile()

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [7]:
# Helper function for formatting the stream nicely
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()


# inputs = {"messages": [("user", "Add 2 and 3 and after that substract 10 from the result.")]}
inputs = {"messages": [("user", "Calsulate 2 + 3 - 7. ")]}
print_stream(app.stream(inputs, stream_mode="values"))

================================ Human Message =================================

Calsulate 2 + 3 - 7. 
================================== Ai Message ==================================
Tool Calls:
  add (4a41b0a2-bcf0-4674-b165-7d8be3e8549f)
 Call ID: 4a41b0a2-bcf0-4674-b165-7d8be3e8549f
  Args:
    a: 2
    b: 3
  substract (673b433e-eef2-47be-8ed5-29d760859beb)
 Call ID: 673b433e-eef2-47be-8ed5-29d760859beb
  Args:
    a: 5
    b: 7
--> add a: 2 b: 3
--> sub a: 5 b: 7
================================= Tool Message =================================
Name: substract

-2
================================== Ai Message ==================================

The result of the calculation 2 + 3 - 7 is -2.
